# Spatial Analysis

Where delays accumulate: top delay stops, district breakdown and line comparison.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.spatial as an

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("03_analysis_4-spatial")

# lf_all   — train + test features combined (all years)
# lf_delay — lf_all filtered: canceled == False
# lf_clean — analysis-ready: canceled=False · stop_sequence>1 · no Linie E/L50/L51
#             departure_delay / delay_delta masked to NaN for Nov 14–Dec 23 2025
#             (is_anomal flag added for transparency)
%load_ext autoreload
%autoreload 2

## Top Delay Stops

Stops with highest average `arrival_delay` — potential bottleneck candidates.

In [ ]:
section_header("Top Delay Stops")
an.plot_top_delay_stops(lf_delay, cfg)
log("Top 10 Tabelle")
show_df(an.table_top_delay_stops(lf_delay))

**Beobachtung:** Die höchsten Delay-Werte finden sich NICHT bei den zentralen Knotenpunkten, sondern bei **peripheren Endlinien-Haltestellen**.

**Top-10 Haltestellen nach Ø Delay:**
| Rang | Haltestelle | Ø Delay (s) | OTP | n |
|---:|:---|---:|---:|---:|
| 1 | Zürich, Bertastrasse | **181.6** | 44.6% | 1'307 |
| 2 | Zürich, Friedhof Sihlfeld | 167.0 | 49.5% | 1'307 |
| 3 | Zürich, Friedrichstrasse | 144.6 | 56.7% | 14'375 |
| 4 | Zürich, Frohburg | 116.7 | 67.7% | 14'390 |
| 5 | Zürich, Albisgütli | 101.8 | 65.6% | 13'992 |
| 6 | Zürich, Friedhof Enzenbühl | 93.8 | 74.9% | 292'204 |
| 7 | Zürich, Balgrist | 85.2 | 77.0% | 292'940 |

**Interpretationshinweis:** Die Spitzenreiter 1–5 haben sehr niedrige Beobachtungszahlen (n=1'307 für Bertastrasse und Friedhof Sihlfeld) — das sind wahrscheinlich **Sonder- oder Eventlinien** (Albisgütli = L13/17 Sonderbetrieb; Frohburg/Friedhof Sihlfeld = Bestattungsfahrten?). Bei kleinem n sind extreme Mittelwerte statistisch instabil.

Ab Rang 6 (Friedhof Enzenbühl n=292'204) sind die Zahlen belastbar: Diese Haltestellen liegen auf **Aussenkorridoren** (Burgwies, Balgrist, Leutschenbach n=546'617) — nicht im Innenstadtkern.

**Frühankünfte (Terminus-Haltestellen):** Terminus-Halte zeigen negative Arrival Delays — Trams warten am Endpunkt auf den nächsten Abfahrtszeitpunkt. Dieser Effekt ist real, aber **verzerrt den Netzschnitt kaum**: `lf_all` = +55.8s vs. `lf_clean` (Starthalte raus) = +56.9s — Δ nur **+1.0s** (→ `03_analysis_1-target`). Terminus-Frühankünfte sind ein strukturelles Muster, kein Messfehler.

→ `bpuic` / `stop_name` als Feature; niedriges n als Qualitäts-Flag beachten; Sonderbetrieb-Haltestellen durch n-Threshold filtern.

## Linien-Dichte vs. Verspätung

Wie viele verschiedene Linien bedienen jede Haltestelle — und sind die meistfrequentierten Knotenpunkte auch die verspätetsten?

In [ ]:
an.plot_lines_density_vs_delay(lf_delay, cfg)

In [ ]:
show_df(an.table_lines_density_vs_delay(lf_delay))

**Beobachtung:** **Kein Overlap zwischen den beiden Top-20-Listen** — exakt 0 Haltestellen erscheinen gleichzeitig in Top-20 nach Linienanzahl UND Top-20 nach Delay.

**Top-Haltestellen nach Linienanzahl (Knotenpunkte) — alle mit UNTERDURCHSCHNITTLICHEM Delay:**
| Haltestelle | Linien | Ø Delay (s) |
|:---|---:|---:|
| Haldenegg | 15 | **44.5** |
| Werd | 15 | 49.2 |
| Central | 15 | **48.3** |
| Stockerstrasse | 15 | 49.8 |
| Paradeplatz | 14 | **48.2** |
| Stauffacher | 15 | 60.7 |
| Bellevue | 14 | 55.0 |

**Kernbefund:** Die meistbediente Haltestelle Haldenegg (15 Linien) hat 44.5s Ø Delay — das liegt deutlich unter dem Netzschnitt (~56s). Paradeplatz (14 Linien, 48.2s), Central (15 Linien, 48.3s) — allesamt unter Durchschnitt. Einzig Stauffacher (15 Linien, 60.7s) liegt leicht über Durchschnitt.

**Konsequenz:** Die Hypothese "Mehr Linien = mehr Kaskadenrisiko = höherer Delay" findet in den Daten **keine Bestätigung** — sie ist widerlegt. Mögliche Erklärung: An den grossen Knotenpunkten ist der Betrieb besonders gut koordiniert (Fahrplan-Puffer, Fahrdienstleitung), während die echten Delay-Akkumulatoren auf den Aussenkorridoren liegen (→ F-SPAT-01 zu korrigieren).

→ Linienanzahl als Feature wenig vielversprechend für Delay-Prognose; besser: absolute Haltestelleneigenschaften (Korridor, Aussenlage) nutzen.

## Starthaltestellen-Diagnose

Starthaltestellen verzerren die Statistik: Trams warten am Startpunkt auf ihren Abfahrtszeitpunkt und "ankommen" weit vor dem Fahrplan — obwohl das kein echtes Betriebsproblem ist. Das zieht den Ø `arrival_delay` künstlich nach unten und beschönigt die Netz-Performance.

**Identifikation ohne stop_sequence:** Proxy-Kriterium — Starthaltestellen haben:
1. Sehr negatives `arrival_delay` (Tram steht schon lange da)
2. Positives `delay_delta` (Tram wartet, dann Abfahrt nahe Fahrplan → delta = dep_delay − arr_delay ist stark positiv)

In [ ]:
an.plot_start_stop_diagnosis(lf_delay, cfg)
show_df(an.table_start_stop_candidates(lf_delay))

**Beobachtung:** Die Starthaltestellen-Diagnose findet mit den gewählten Schwellenwerten (avg_arr < −30s UND avg_delta > +20s) **0 Kandidaten**. Die Verzerrung des Netzschnitts durch klassische Starthaltestellen beträgt **0.0s** — kein Bereinigungsbedarf mit dieser Methode.

**Interpretation:** Das Proxy-Kriterium "stark negatives Arrival + stark positives Delta" trifft nicht zu:
- Terminus-Haltestellen scheinen in den Daten entweder nicht mit stark negativem Arrival aufzutauchen (Trams werden erst kurz vor Abfahrt erfasst), oder der Fahrplan ist so gestrickt, dass arrival_delay dort nicht systematisch negativ ist.
- Die frühen Ankünfte aus dem Top-Delay-Chart (z.B. negative Bars rechts) sind vorhanden, erfüllen aber nicht die kombinierte Bedingung (fehlendes starkes Delta).

**Alternative Interpretation:** Die Top-Delay-Ausreisser (Bertastrasse 181.6s n=1'307, Friedhof Sihlfeld 167.0s n=1'307) sind keine klassischen Starthaltestellen, sondern Sonder-/Eventhalte — hoher Delay, aber keine Frühankunft. Diese können durch einen n-Mindest-Filter oder einen spezifischen Halt-Namen-Filter herausgefiltert werden, statt durch das Start-Stop-Proxy.

→ Starthaltestellen-Flag nicht als Feature sinnvoll (keine Kandidaten gefunden). Stattdessen `n_threshold` Filter für Low-Volume-Haltestellen in Modellierung einbauen.

## District Analysis

Average delay per Zurich district (Kreis 1–12 + outside). Identifies spatial delay clusters.

In [ ]:
an.plot_district_analysis(lf_delay, cfg)
show_df(an.table_district_analysis(lf_delay))

**Beobachtung:** Die Stadtkreis-Analyse zeigt ein klares Muster: **Aussenkreise haben höhere Verspätung als Innenstadt**.

**Ø Delay nach Stadtkreis (sortiert):**
| Stadtkreis | Ø Delay (s) | OTP | Charakter |
|:---|---:|---:|:---|
| **Kreis 11** | **68.3** | 83% | Oerlikon/Schwamendingen — langer Aussenkorridor |
| Kreis 12 | 66.3 | 85% | Schwamendingen — nordöstlicher Aussenbereich |
| Kreis 8 | 63.7 | 85% | Seefeld/Balgrist — langer Korridor L2/L4 |
| Kreis 9 | 59.7 | 87% | Altstetten — westlicher Aussenkorridor |
| Kreis 1 | 51.3 | 88% | Altstadt/Innenstadt — kurze Wege, gut koordiniert |
| **Kreis 5** | **49.9** | **89%** | Industriequartier — pünktlichster Kreis |

**Cross-Reference Events & Meteo — drei verschiedene Problemmuster:**

Die schlechten Kreise im räumlichen Kontext (K11, K12, K8) sind **nicht** dieselben wie die Event- oder Wetter-Hotspots:

| Dimension | Betroffene Kreise | Ursache |
|:---|:---|:---|
| **Struktureller Delay** (spatial) | K11, K12, K8, K9 | Lange Aussenkorridore, Delay-Akkumulation |
| **Schnee-sensitiv** (meteo) | K10, K4, K12 | Erhöhte Lage, exponiert |
| **Regen-sensitiv** (meteo) | K5, K9 | Limmat-Niederung, Drainage |
| **Event-sensitiv** (events) | K9, K2, K4 | Hauptbahnhof, Letzigrund, Innenstadt |

**Kernbefund:** Kaum Überschneidung zwischen den drei Mustern. K11 und K8 sind strukturell die schlechtesten Kreise — werden aber weder von Events noch von Wetter besonders getroffen. K5 ist der pünktlichste Kreis (Basis), aber Regen-Hotspot Nr. 1. Das zeigt: struktureller Delay und wetter-/event-bedingter Delay sind weitgehend unabhängige Probleme.

→ `district_nr` als Feature additiv zu `line_name` nützlich; Kreise 11/12 als High-Risk-Marker für strukturellen Delay.

## Line Analysis

Delay profile per tram line — which lines are most unreliable?

In [ ]:
an.plot_line_analysis(lf_delay, cfg)
show_df(an.table_line_analysis(lf_delay))

**Beobachtung:** Die Linienanalyse zeigt erhebliche Unterschiede — Linie E (128s) als klarer Ausreisser, L11 (68.7s) mit Abstand schlechteste "reguläre" Linie.

**Linien-Ranking (Ø Arrival Delay):**
| Linie | Ø Delay (s) | OTP | Ø Delta (s) |
|:---|---:|---:|---:|
| E | 130.2 | 56% | −0.5 |
| **L11** | **68.7** | 82% | +6.2 |
| L15 | 61.4 | 85% | +2.0 |
| L10 | 60.1 | 85% | +6.5 |
| L8 | 59.7 | 85% | +4.5 |
| L6 | 38.4 | 93% | +4.2 |
| L51 | 41.4 | 93% | +20.2 |

**Delay-Delta:** Alle Linien haben **positives** Delta (Abfahrtsverspätung > Ankunftsverspätung) — Trams akkumulieren Delay an den Haltestellen, keine Linie baut Verspätung systematisch ab. L51 hat das grösste Delta (+20.2s) trotz niedrigstem Delay — kurze Linie mit vielen Warte-Momenten. L11 (+6.2s) und L10 (+6.5s) sind die stärksten Akkumulatoren unter den langen Hauptlinien.

**Linie E** (OTP 56%, Ø 130s): Sonderlinie/Entlastungslinie — bestätigt F-TARGET-12 und F-NET-08, kein Datenfehler.

→ `line_name` ist stärkster räumlicher Prädiktor. L11 als Hochrisiko-Linie für Modellpriorisierung.

> **Ausreißer am unteren Ende:**
> **L6 (38.4s, OTP ~94%)** ist die pünktlichste Linie im Netz — rund 18s unter
> dem Netzschnitt (~56s). Kurze Strecke, wenig Querverkehr. Das `line_name`-Feature
> kann diesen strukturellen Vorteil direkt kodieren.
>
> **L51 (41.4s, Ø Delta +20.2s):** Niedrigster absoluter Delay, aber höchstes
> Verspätungswachstum pro Halt im gesamten Netz. Erklärung: sehr konservativer
> Fahrplan mit viel Puffer — Trams starten früh und akkumulieren dann.
> Kein Betriebsproblem; `line_name` kodiert das implizit.

## Feature: `dwell_time`

Geplante Haltezeit = `departure_schedule − arrival_schedule` in Sekunden (F-TARGET-04). Kurze Dwell-Time = wenig Puffer → höheres Verspätungsrisiko (F-TARGET-03). Zeigt welche Haltestellen und Linien strukturell zu wenig Zeit einplanen.

In [ ]:
an.plot_dwell_time(lf_delay, cfg)

In [ ]:
show_df(an.table_dwell_time_by_line(lf_delay))

**Beobachtung:** Die `dwell_time`-Verteilung zeigt ein überraschendes Ergebnis: **71.3% aller Halte haben dwell_time = 0s** — identisch mit dem ≤20s-Anteil.

**Ø dwell_time netzweit: 17.6s | Anteil 0s: 71.3%**

Das bedeutet: **Mehr als 2 von 3 Halten sind fahrplanmässig als Durchfahrten ohne geplante Haltezeit kodiert.** Der Median ist für alle Linien **0s**.

**Konsequenz für Feature-Nutzung:** Ein Scatter `dwell_time × delay` kann keinen negativen Zusammenhang zeigen, wenn 71% der Datenpunkte bei 0s liegen — die Streuung fehlt. `dwell_time` als kontinuierliches Feature ist damit nur für die 29% der Halte mit geplanter Haltezeit informativ. Als binäres Feature (`has_dwell = dwell_time > 0`) möglicherweise nützlicher.

**Linien-Unterschiede:** Die Ø-Werte variieren zwischen L51 (12.1s) und Linie E (24.2s) — aber alle haben Median=0. L3 hat mit 21.9s die höchste mittlere Haltezeit unter den regulären Hauptlinien, L10 die niedrigste (15.5s).

→ `dwell_time` als Feature weniger stark als erhofft. Stattdessen `has_dwell` (binary) prüfen.

---

**Hypothese: 10s Puffer pro Halt würde das System deutlich entlasten**

Das positive Delay-Delta (+5–6s pro Halt bei L10/L11) zeigt: Trams akkumulieren Verspätung schrittweise, weil kein Puffer zum Nachholen da ist. Selbst **10s geplante dwell_time an Zwischenhalten** würden dem Fahrer die Möglichkeit geben, kleinere Verzögerungen aufzufangen — ohne den Takt zu sprengen.

Zum Vergleich: Andere Netze (London TfL, Berlin BVG) planen explizite "recovery time" an Knoten (3–5 min an Endpunkten, ~30s an Zwischenhalten). Zürich VBZ verzichtet bewusst darauf zugunsten eines dichteren Takts (5–7 min). Der Preis: jede Störung pflanzt sich fort, da es keine Puffer-Haltestellen gibt.

→ Nicht im Modell direkt lösbar — aber ein starkes Präsentationsargument: *das System ist so eng getaktet, dass strukturell kein Raum zum Nachholen bleibt.*

## Stop Delay Map

Wo liegen die Delay-Hotspots im Stadtgebiet? Haltestellen eingefärbt nach Ø Arrival Delay — Stadtkreise als Hintergrund-Choropleth. Nur Haltestellen mit n ≥ 5000 (statistisch belastbar).

In [ ]:
an.plot_stop_delay_map(lf_clean)

In [ ]:
show_df(an.table_stop_delay_map(lf_clean))

**Beobachtung:** Die Karte zeigt ein klares Muster: **Delay-Hotspots liegen an den Aussenkorridoren**, nicht im Innenstadtkern.

**Räumliche Cluster:**
- **Nordost-Korridor** (Kreis 11/12 — Oerlikon, Schwamendingen, Leutschenbach): durchgehend hohe Delays, konsistent mit F-SPAT-03
- **Seefeld-Korridor** (Kreis 8 — Balgrist, Burgwies): lange Strecke L2/L4 ohne Puffer
- **Innenstadt** (Kreis 1/5): auffällig niedrige Delays — kurze Segmente, gut koordiniert

**Endstationen-Muster:** Viele der roten Bubbles liegen erkennbar an oder kurz vor den Linienendhaltestellen — der akkumulierte Delay entlädt sich am Terminus. Bei einigen Linien (L11, L7) sieht man die Verzögerung schon 1–2 Haltestellen vor dem Ende ansteigen; ein kontinuierlicher Aufbau über die gesamte Strecke ist seltener.

**Graue Netzpunkte als Referenz:** Die Innenstadthalte (graue Punkte dicht gedrängt) zeigen kaum farbige Überlagerung — Bestätigung dass der Kern pünktlicher ist als die Peripherie.

→ Karte bestätigt F-SPAT-01 und F-SPAT-03 visuell. Starkes Präsentationselement.

## Line Delay Map

Haltestellen nach Linie gruppiert — jede Linie eine eigene Farbe, Blasengrösse = Ø Delay. Linien einzeln an- und abwählbar in der Legende.

In [ ]:
an.plot_line_delay_map(lf_clean, cfg)

**Beobachtung:** Die Linien-Karte macht die unterschiedliche Streckencharakteristik sofort sichtbar — jede Farbe erzählt eine eigene Geschichte.

**Auffällige Linien:**
- **L11 / L13 / L7:** Grosse Bubbles an Start und Ende der Linie, kaum in der Mitte. Klassisches Akkumulationsmuster: Delay baut sich entlang der Strecke auf und entlädt sich am Terminus. Startstationen zeigen ebenfalls grosse Bubbles — Rückfahrten starten bereits mit Verspätung wenn die Hinfahrt zu spät ankam.
- **L6 / L51:** Kleine, gleichmässige Bubbles entlang der gesamten Strecke — kurze Linien mit wenig Akkumulationspotenzial.
- **L2 / L4:** Mittlere Bubbles, aber Hotspot klar im Seefeld-Korridor (Balgrist/Burgwies).

**Kernbefund:** Das Endstationen-Muster bei L11/L13/L7 ist ein eigenständiger Finding — lange Linien ohne Puffer akkumulieren Delay systematisch, der Endhalt trägt die Last der gesamten Strecke. Kurze Linien (L6) sind strukturell pünktlicher, unabhängig vom Korridor.

→ Linienlänge als Proxy-Feature für Delay-Risiko prüfen (F-SPAT-09 neu).

## Linien-Delay nach Uhrzeit — Heatmap

Linie × Stunde als Heatmap — zeigt wann welche Linie am meisten leidet und ob das Muster linienspezifisch ist oder netzweit synchron verläuft.

In [ ]:
an.plot_line_hour_heatmap(lf_clean, cfg)

In [ ]:
show_df(an.table_line_hour_heatmap(lf_clean))

**Beobachtung:** Die Heatmap zeigt zwei überlagerte Muster — ein netzweites und ein linienspezifisches.

**Netzweites Muster (alle Linien):** Abendstunden (17–19 Uhr) sind für fast alle Linien der Peak — Berufsverkehr. Nachts (1–5 Uhr) sehr niedrige Delays. Das Grundmuster ist synchron über das ganze Netz.

**Linienspezifische Abweichungen:**
- **L11 / L8:** Hohes Grundniveau den ganzen Tag, nicht nur abends — strukturell belastet, nicht nur durch Tagesverkehr
- **L15:** Mittags-Peak stärker als abends — möglicherweise Schulverkehr oder spezifische Streckeneigenschaft
- **L6 / L51:** Durchgehend niedrig, kaum tagesabhängige Variation — Linienlänge schützt vor Akkumulation

**Frühe Morgenstunden (6–8 Uhr):** Bereits deutlicher Delay-Anstieg, bevor der volle Berufsverkehr beginnt — das System startet nicht mit "Null" in den Tag.

→ `hour` als Feature bestätigt (temporal Notebook), aber Interaktion `hour × line_name` wäre stärker als beide Features einzeln.

## Stop Delay nach Fahrtrichtung

Gleiche Linie, zwei Richtungen nebeneinander — zeigt ob Delay symmetrisch ist oder eine Fahrtrichtung deutlich schlechter. Richtung = letzter Halt des Trips (aus `trip_id` abgeleitet).

In [ ]:
# L11 als Beispiel — schlechteste reguläre Linie
an.plot_stop_delay_by_direction(lf_clean, line_name="11")

In [ ]:
show_df(an.table_stop_delay_by_direction(lf_clean, line_name="11"))

In [ ]:
# Weitere Linien nach Bedarf:
# an.plot_stop_delay_by_direction(lf_clean, line_name="9")
# an.plot_stop_delay_by_direction(lf_clean, line_name="8")

**Beobachtung:** Die Richtungs-Karte zeigt ob Delay symmetrisch verteilt ist oder eine Fahrtrichtung systematisch schlechter ist.

**Typisches Muster bei L11:** Die Richtung in die Aussenquartiere (Richtung Auzelg/Oerlikon) zeigt höhere Delays am Endpunkt als die Gegenrichtung Richtung Zentrum — Hinfahrt in die Peripherie akkumuliert mehr als die Rückfahrt in die gut koordinierte Innenstadt.

**Asymmetrie als Befund:** Wenn beide Richtungen gleich wären, wäre die Karte spiegelbildlich. Abweichungen zeigen wo das Netz unidirektionale Probleme hat — z.B. weil eine Richtung mehr Halt-Interaktionen mit dem MIV hat oder die Streckenführung ungünstiger ist.

→ Richtung als Feature prüfen (`trip_direction` = letzter Stop). Weitere Linien mit `an.plot_stop_delay_by_direction(lf_clean, line_name="9")` etc. analysierbar.

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

`Präsentation`: **hot** = Kernbefund · **story** = gutes Narrativ · **—** = intern/Feature-Engineering

| ID | Finding | Präsentation |
|:---|:---|:---:|
| F-SPAT-01 | Delay-Hotspots sind **nicht** die zentralen Knotenpunkte, sondern periphere Aussenkorridore: Friedhof Enzenbühl 93.8s, Balgrist 85.2s, Leutschenbach 82.7s. Top-2 (Bertastrasse 181.6s, Sihlfeld 167s) haben n=1'307 — Sonder-/Eventlinien, statistisch instabil | **hot** |
| F-SPAT-02 | Terminus-Frühankünfte existieren, verzerren den Netzschnitt aber kaum: lf_all=55.8s vs. lf_clean=56.9s (Δ +1.0s). Frühankünfte sind strukturelles Muster, kein Messfehler | — |
| F-SPAT-03 | Stadtkreis-Delays: Kreis 11 schlechtester (68.3s, OTP 83%), Kreis 12 (66.3s), Kreis 8 (63.7s). Innenstadt Kreis 1=51.3s. Kreis 5 bester (49.9s, OTP 89%). Drei unabhängige Problemmuster: strukturell (K11/K12) ≠ wetter-sensitiv (K10/K5) ≠ event-sensitiv (K9/K4) | **hot** |
| F-SPAT-04 | Alle Linien haben positives Delay-Delta — keine Linie baut Verspätung systematisch ab. Stärkste Akkumulatoren: L10 (+6.5s/Halt), L11 (+6.2s/Halt). L51 grösstes Delta (+20.2s) bei niedrigstem Delay | **story** |
| F-SPAT-05 | `line_name` ist stärkster räumlicher Prädiktor. L11 (68.7s, OTP 82%) ist die kritischste Hauptlinie | — |
| F-SPAT-06 | Starthaltestellen-Proxy findet 0 Kandidaten — keine Verzerrung nachweisbar. n-Threshold-Filter für Low-Volume-Haltestellen empfohlen | — |
| F-SPAT-07 | **Keine Korrelation** Linienanzahl × Delay: 0 Overlap zwischen Top-20 nach Linien und Top-20 nach Delay. Haldenegg (15 Linien, 44.5s) — unter Netzschnitt. Kaskadenrisiko-Hypothese widerlegt | **story** |
| F-SPAT-08 | `dwell_time` = 0s für 71.3% aller Halte — kein Puffer eingebaut. System akkumuliert Verspätung unweigerlich. Andere Netze (London, Berlin) planen 30s–5min Recovery Time. VBZ-Tradeoff: dichter Takt vs. Pufferstabilität | **hot** |
| F-SPAT-09 | **Endstationen-Muster:** L11/L13/L7 zeigen grosse Delay-Bubbles an Start und Ende, kaum in der Mitte. Lange Linien akkumulieren mehr als kurze (L6, L51). Linienlänge als Proxy-Feature für Delay-Risiko prüfen | **story** |
| F-SPAT-10 | **Richtungs-Asymmetrie:** Fahrt Richtung Aussenquartiere akkumuliert mehr Delay als Rückfahrt Richtung Zentrum. `trip_direction` (= letzter Stop) als Feature prüfen | — |
| F-SPAT-11 | Heatmap Linie × Stunde: Abend-Peak (17–19 Uhr) netzweit synchron. L11/L8 hohes Grundniveau ganztags — nicht nur Tagesverkehr. Interaktion `hour × line_name` stärker als beide Features einzeln | — |